In [15]:
import pandas as pd

DATASET_PATH = "../data/raw/online_retail_II.xlsx"

sheets = pd.read_excel(
    DATASET_PATH,
    sheet_name=None,
    engine="openpyxl"
)

for name, df in sheets.items():
    print(f"\nSheet: {name}")
    print(f"Shape: {df.shape}")
    print("\nColumn types:")
    print(df.dtypes)


Sheet: Year 2009-2010
Shape: (525461, 8)

Column types:
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object

Sheet: Year 2010-2011
Shape: (541910, 8)

Column types:
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object


In [ ]:
# Combine the two years into one dataset
retail_df = pd.concat(
    [
        df.assign(SourceYear=sheet_name)
        for sheet_name, df in sheets.items()
    ],
    ignore_index=True
)

print("BUYERPULSE — COMBINED DATASET")
print("=" * 45)

print(f"Total records: {len(retail_df):,}")
print(f"Total columns: {retail_df.shape[1]}")

print("\nColumn names:")
print(retail_df.columns.tolist())

print("\nFirst five records:")
display(retail_df.head())

print("\nDataset memory usage:")
print(
    f"{retail_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

In [17]:
# Step 2.6 — Missing Value Analysis

missing_count = retail_df.isna().sum()
missing_percentage = (missing_count / len(retail_df)) * 100

missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percentage (%)": missing_percentage.round(2)
})

missing_summary = missing_summary.sort_values(
    by="Missing Count",
    ascending=False
)

print("BUYERPULSE — MISSING VALUE ANALYSIS")
print("=" * 45)

display(missing_summary)

print(f"\nTotal records: {len(retail_df):,}")

print(
    f"Records with at least one missing value: "
    f"{retail_df.isna().any(axis=1).sum():,}"
)

BUYERPULSE — MISSING VALUE ANALYSIS


,Missing Count,Missing Percentage (%)
Customer ID,243007,22.77
Description,4382,0.41
Invoice,0,0.00
StockCode,0,0.00
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Country,0,0.00
SourceYear,0,0.00



Total records: 1,067,371
Records with at least one missing value: 243,007


In [ ]:
# Step 2.7 — Transaction Anomaly Analysis

invoice_str = retail_df["Invoice"].astype("string").str.strip()

cancelled_invoice = invoice_str.str.upper().str.startswith(
    "C", na=False
)

negative_quantity = retail_df["Quantity"] < 0
zero_quantity = retail_df["Quantity"] == 0

negative_price = retail_df["Price"] < 0
zero_price = retail_df["Price"] == 0

anomaly_summary = pd.DataFrame({
    "Anomaly": [
        "Cancelled invoice (starts with C)",
        "Negative quantity",
        "Zero quantity",
        "Negative price",
        "Zero price"
    ],
    "Affected Rows": [
        cancelled_invoice.sum(),
        negative_quantity.sum(),
        zero_quantity.sum(),
        negative_price.sum(),
        zero_price.sum()
    ]
})

anomaly_summary["Percentage (%)"] = (
    anomaly_summary["Affected Rows"]
    / len(retail_df) * 100
).round(2)

print("BUYERPULSE — TRANSACTION ANOMALIES")
print("=" * 45)

display(anomaly_summary)

print("\nOverlap: cancelled invoice + negative quantity")
print((cancelled_invoice & negative_quantity).sum())

print("\nExamples of cancelled invoices:")
display(retail_df.loc[cancelled_invoice].head(5))

In [ ]:
# Step 2.7.2 — Investigate cancellation and quantity exceptions

exceptions = {
    "C invoice with non-negative quantity":
        cancelled_invoice & ~negative_quantity,

    "Negative quantity without C invoice":
        negative_quantity & ~cancelled_invoice,

    "Negative price":
        negative_price,

    "Zero price":
        zero_price,
}

for name, mask in exceptions.items():
    subset = retail_df.loc[mask]

    print(f"\n{name}: {len(subset):,} rows")
    print("Most common StockCodes:")
    print(subset["StockCode"].value_counts().head(10))

    print("\nExample records:")
    display(subset.head(5))

### Transaction Anomaly Investigation — Preliminary Findings

**Dataset:** UCI Online Retail II (1,067,371 transaction lines)

The initial audit identified 19,494 cancellation-marked rows, 22,950 negative-quantity rows, 6,202 zero-price rows, and 5 negative-price rows.

Of the 19,494 cancellation-marked rows, 19,493 also have negative quantities. However, 3,457 negative-quantity rows do not have cancellation-marked invoice identifiers.

Inspection of exceptional records revealed:

- A cancellation-marked invoice with a positive quantity and missing customer ID.
- Negative-quantity transactions that appear consistent with inventory corrections.
- Five negative-price records explicitly labeled as bad-debt adjustments.
- Zero-price records with mixed characteristics requiring additional investigation.

**Preliminary conclusion:** Identifying completed customer purchases requires multiple validation conditions, not just checking invoice prefixes. Cleaning rules must be applied consistently, and excluded records should be counted for reproducibility.

**Status:** Exploratory findings; cleaning policy pending validation.

In [ ]:
# Step 2.7.3 — Zero-Price Transaction Investigation

zero_price_df = retail_df.loc[zero_price].copy()

zero_price_summary = {
    "Total zero-price rows": len(zero_price_df),
    "Positive quantity": (zero_price_df["Quantity"] > 0).sum(),
    "Negative quantity": (zero_price_df["Quantity"] < 0).sum(),
    "Missing Customer ID": zero_price_df["Customer ID"].isna().sum(),
    "Known Customer ID": zero_price_df["Customer ID"].notna().sum(),
    "Cancelled invoice": cancelled_invoice.loc[zero_price_df.index].sum(),
}

print("BUYERPULSE — ZERO-PRICE INVESTIGATION")
print("=" * 45)

for category, count in zero_price_summary.items():
    print(f"{category}: {count:,}")

print("\nMost common descriptions:")
display(
    zero_price_df["Description"]
    .value_counts(dropna=False)
    .head(15)
)

print("\nExamples: positive quantity + known customer + zero price")

display(
    zero_price_df.loc[
        (zero_price_df["Quantity"] > 0)
        & zero_price_df["Customer ID"].notna()
    ].head(10)
)

### Zero-Price Transaction Analysis

The dataset contains **6,202 zero-price transaction lines**, representing approximately 0.58% of all records.

Of these:
- 2,745 have positive quantities.
- 3,457 have negative quantities.
- 6,131 have missing customer identifiers.
- 71 have identifiable customers.
- None have cancellation-marked invoice identifiers.

All 4,382 records with missing product descriptions also belong to the zero-price group.

The zero-price group includes descriptions indicating possible stock corrections, such as `check`, `damages`, `missing`, and `adjustment`, alongside recognizable retail products.

**Interpretation:** Zero-price lines are heterogeneous and should not automatically be interpreted as completed purchases. However, zero-price items may coexist with paid items on genuine customer invoices.

**Proposed treatment:** Define purchases at the invoice level using qualifying positive-quantity, positive-price sales lines. Treat zero-price lines separately when calculating monetary features, and examine their relationship to otherwise valid invoices.

**Status:** Preliminary policy; pending invoice-level validation and cleaning implementation.

In [21]:
# Step 2.7.4 — Invoice-Level Zero-Price Analysis (Corrected)

invoice_keys = ["SourceYear", "Invoice"]

known_customer_zero = retail_df.loc[
    (retail_df["Price"] == 0)
    & (retail_df["Quantity"] > 0)
    & retail_df["Customer ID"].notna()
].copy()

paid_lines = retail_df.loc[
    retail_df["Customer ID"].notna()
    & (retail_df["Quantity"] > 0)
    & (retail_df["Price"] > 0)
    & ~cancelled_invoice,
    invoice_keys
].drop_duplicates()

zero_invoices = known_customer_zero[
    invoice_keys
].drop_duplicates()

invoice_comparison = zero_invoices.merge(
    paid_lines.assign(HasPaidLine=True),
    on=invoice_keys,
    how="left",
    validate="one_to_one"
)

# Explicitly convert to a proper Boolean dtype
has_paid_line = (
    invoice_comparison["HasPaidLine"]
    .fillna(False)
    .astype(bool)
)

with_paid = int(has_paid_line.sum())
without_paid = int((~has_paid_line).sum())

# Validate that the categories account for every invoice
assert with_paid + without_paid == len(zero_invoices)
assert with_paid >= 0 and without_paid >= 0

print("BUYERPULSE — ZERO-PRICE INVOICE ANALYSIS")
print("=" * 45)

print(
    "Positive-quantity zero-price rows with known customers:",
    len(known_customer_zero)
)

print("Distinct invoices:", len(zero_invoices))
print("Invoices also containing paid lines:", with_paid)
print("Invoices without qualifying paid lines:", without_paid)

BUYERPULSE — ZERO-PRICE INVOICE ANALYSIS
Positive-quantity zero-price rows with known customers: 71
Distinct invoices: 62
Invoices also containing paid lines: 56
Invoices without qualifying paid lines: 6


### Invoice-Level Validation of Zero-Price Records

Among 71 positive-quantity, zero-price transaction lines with identifiable customers, we identified 62 distinct invoices.

- **56 invoices (90.3%)** also contain qualifying paid transaction lines.
- **6 invoices (9.7%)** have no qualifying paid transaction lines.

This supports defining valid purchases at the invoice level rather than discarding entire invoices whenever a zero-price line is present.

**Proposed policy:** Retain qualifying paid purchase lines, exclude zero-price lines from monetary calculations, and do not count zero-price-only invoices as paid purchases.

The corrected analysis includes an assertion verifying that the two invoice categories account for all 62 invoices.

In [ ]:
# Step 2.8 — Duplicate Transaction Analysis

# Exclude SourceYear because it is metadata added by us
original_columns = [
    col for col in retail_df.columns
    if col != "SourceYear"
]

duplicate_mask = retail_df.duplicated(
    subset=original_columns,
    keep="first"
)

duplicate_rows = int(duplicate_mask.sum())

print("BUYERPULSE — DUPLICATE RECORD ANALYSIS")
print("=" * 45)

print(f"Total records: {len(retail_df):,}")
print(f"Duplicate rows: {duplicate_rows:,}")
print(f"Duplicate percentage: {duplicate_rows / len(retail_df) * 100:.2f}%")
print(f"Unique rows: {len(retail_df) - duplicate_rows:,}")

print("\nExamples of duplicated records:")

display(
    retail_df.loc[
        retail_df.duplicated(
            subset=original_columns,
            keep=False
        )
    ].sort_values(
        by=["InvoiceDate", "Invoice"]
    ).head(12)
)

### Duplicate Transaction Investigation

An exact-duplicate analysis across the eight original transaction columns identified **34,335 repeated rows beyond their first occurrence**, representing **3.22%** of the 1,067,371 transaction lines.

A hypothetical exact deduplication would leave 1,033,036 rows.

Repeated records were observed within individual invoices, including invoice `489517`, where identical product lines appear multiple times.

**Interpretation:** These repeated records may reflect duplicate data entries or legitimate repeated invoice lines. The dataset does not contain unique invoice-line identifiers, preventing definitive classification based on equality alone.

**Modeling implications:**
- Purchase frequency should count qualifying invoices rather than product rows.
- Duplicate handling may influence monetary value and product quantity features.
- The effect of deduplication should be documented and, where feasible, tested through sensitivity analysis.

**Status:** Exploratory finding. No records have been removed. Deduplication policy pending further investigation.

In [23]:
# Step 2.8.1 — Duplicate Distribution by Source Year

original_columns = [
    col for col in retail_df.columns
    if col != "SourceYear"
]

# Duplicates within the same source worksheet
within_year_duplicates = retail_df.duplicated(
    subset=["SourceYear"] + original_columns,
    keep="first"
)

# Duplicates across both worksheets
global_duplicates = retail_df.duplicated(
    subset=original_columns,
    keep="first"
)

summary = (
    retail_df.assign(
        IsDuplicate=within_year_duplicates
    )
    .groupby("SourceYear")
    .agg(
        TotalRows=("Invoice", "size"),
        DuplicateRows=("IsDuplicate", "sum")
    )
)

summary["DuplicatePercentage"] = (
    summary["DuplicateRows"]
    / summary["TotalRows"] * 100
).round(2)

print("BUYERPULSE — DUPLICATES BY SOURCE YEAR")
print("=" * 45)

display(summary)

print(
    "\nWithin-year duplicates:",
    int(within_year_duplicates.sum())
)

print(
    "Global duplicates:",
    int(global_duplicates.sum())
)

print(
    "Additional cross-year duplicate matches:",
    int(global_duplicates.sum() - within_year_duplicates.sum())
)

BUYERPULSE — DUPLICATES BY SOURCE YEAR


,TotalRows,DuplicateRows,DuplicatePercentage
SourceYear,,,
Year 2009-2010,525461,6865,1.31
Year 2010-2011,541910,5268,0.97



Within-year duplicates: 12133
Global duplicates: 34335
Additional cross-year duplicate matches: 22202


In [ ]:
# Step 2.8.2 — Investigate cross-year duplicate records

original_columns = [
    col for col in retail_df.columns
    if col != "SourceYear"
]

first_year = retail_df.loc[
    retail_df["SourceYear"] == "Year 2009-2010"
]

second_year = retail_df.loc[
    retail_df["SourceYear"] == "Year 2010-2011"
]

# Identify exact records present in both worksheets
cross_year_matches = second_year.merge(
    first_year[original_columns].drop_duplicates(),
    on=original_columns,
    how="inner"
)

print("BUYERPULSE — CROSS-YEAR DUPLICATE INVESTIGATION")
print("=" * 45)

print("Matching rows in second worksheet:",
      f"{len(cross_year_matches):,}")

print("Distinct matching invoices:",
      cross_year_matches["Invoice"].nunique())

print("\nDate range of matching records:")
print("Earliest:", cross_year_matches["InvoiceDate"].min())
print("Latest:", cross_year_matches["InvoiceDate"].max())

print("\nMatching records by month:")
display(
    cross_year_matches.groupby(
        cross_year_matches["InvoiceDate"].dt.to_period("M")
    ).size()
)

print("\nExample cross-year matching records:")
display(cross_year_matches.head(10))

### Cross-Year Dataset Overlap Investigation

An exact-record comparison between the `Year 2009-2010` and `Year 2010-2011` worksheets revealed overlapping transaction records.

**Observed findings:**

- 22,523 rows in the second worksheet matched records in the first worksheet.
- These matching records involved 1,088 distinct invoice identifiers.
- Matching transaction timestamps ranged from December 1, 2010 at 08:26 to December 9, 2010 at 20:01.
- The earlier global duplicate analysis identified 22,202 additional duplicate occurrences when combining the worksheets.

The difference between the two counts reflects their different counting methods and requires careful interpretation.

**Implication:** Combining both worksheets without accounting for repeated transactions could inflate monetary and quantity-based customer features.

**Next step:** Investigate worksheet date coverage and duplicate multiplicities before choosing a deduplication policy.

**Status:** Cross-year overlap confirmed; final cleaning strategy pending validation.

In [25]:
# Step 2.8.3 — Worksheet Date Coverage

print("BUYERPULSE — WORKSHEET DATE COVERAGE")
print("=" * 45)

date_summary = (
    retail_df.groupby("SourceYear")
    .agg(
        TotalRows=("Invoice", "size"),
        EarliestDate=("InvoiceDate", "min"),
        LatestDate=("InvoiceDate", "max"),
        UniqueInvoices=("Invoice", "nunique")
    )
)

display(date_summary)

# Inspect records around the year boundary
boundary_records = retail_df.loc[
    retail_df["InvoiceDate"].between(
        "2010-12-01",
        "2010-12-10",
        inclusive="left"
    )
]

boundary_summary = (
    boundary_records.groupby([
        "SourceYear",
        boundary_records["InvoiceDate"].dt.date
    ])
    .size()
    .rename("TransactionRows")
)

print("\nDaily transaction counts during overlap:")
display(boundary_summary)

BUYERPULSE — WORKSHEET DATE COVERAGE


,TotalRows,EarliestDate,LatestDate,UniqueInvoices
SourceYear,,,,
Year 2009-2010,525461,2009-12-01 07:45:00,2010-12-09 20:01:00,28816
Year 2010-2011,541910,2010-12-01 08:26:00,2011-12-09 12:50:00,25900



Daily transaction counts during overlap:


SourceYear      InvoiceDate
Year 2009-2010  2010-12-01     3108
                2010-12-02     2109
                2010-12-03     2202
                2010-12-05     2725
                2010-12-06     3878
                2010-12-07     2963
                2010-12-08     2647
                2010-12-09     2891
Year 2010-2011  2010-12-01     3108
                2010-12-02     2109
                2010-12-03     2202
                2010-12-05     2725
                2010-12-06     3878
                2010-12-07     2963
                2010-12-08     2647
                2010-12-09     2891
Name: TransactionRows, dtype: int64

In [31]:
# Step 2.8.4 — Exact Cross-Year Overlap Verification
# BuyerPulse — Missing-value-safe comparison

import pandas as pd

print("BUYERPULSE — EXACT OVERLAP VERIFICATION")
print("=" * 45)

# 1. Original transaction columns
original_columns = [
    col for col in retail_df.columns
    if col != "SourceYear"
]

# 2. Define overlapping period
overlap_start = pd.Timestamp("2010-12-01")
overlap_end = pd.Timestamp("2010-12-10")

overlap_df = retail_df.loc[
    (retail_df["InvoiceDate"] >= overlap_start)
    & (retail_df["InvoiceDate"] < overlap_end)
]

# 3. Separate worksheets
first_year = overlap_df.loc[
    overlap_df["SourceYear"] == "Year 2009-2010",
    original_columns
].copy()

second_year = overlap_df.loc[
    overlap_df["SourceYear"] == "Year 2010-2011",
    original_columns
].copy()

# 4. Count occurrences of each distinct record
# dropna=False ensures missing values are included.

first_counts = (
    first_year
    .groupby(original_columns, dropna=False)
    .size()
    .reset_index(name="FirstCount")
)

second_counts = (
    second_year
    .groupby(original_columns, dropna=False)
    .size()
    .reset_index(name="SecondCount")
)

# 5. Merge by all original transaction columns.
# Pandas merge matches null keys against null keys.

comparison = first_counts.merge(
    second_counts,
    on=original_columns,
    how="outer",
    validate="one_to_one",
    indicator=True
)

comparison["FirstCount"] = (
    comparison["FirstCount"]
    .fillna(0)
    .astype(int)
)

comparison["SecondCount"] = (
    comparison["SecondCount"]
    .fillna(0)
    .astype(int)
)

# 6. Find actual differences in occurrence counts
differences = comparison.loc[
    comparison["FirstCount"] != comparison["SecondCount"]
]

# 7. Display results
print(f"First worksheet rows: {len(first_year):,}")
print(f"Second worksheet rows: {len(second_year):,}")

print(
    "Distinct records with different counts:",
    f"{len(differences):,}"
)

print("\nMerge classification:")
print(comparison["_merge"].value_counts())

if differences.empty:
    print(
        "\nRESULT: Overlapping records are identical, "
        "including multiplicities."
    )
else:
    print("\nRESULT: Differences detected.")
    display(differences.head(10))

# 8. Validate count preservation
assert comparison["FirstCount"].sum() == len(first_year)
assert comparison["SecondCount"].sum() == len(second_year)

if differences.empty:
    print("\nBuyerPulse overlap validation PASSED!")
else:
    print("\nBuyerPulse overlap validation FAILED!")

BUYERPULSE — EXACT OVERLAP VERIFICATION
First worksheet rows: 22,523
Second worksheet rows: 22,523
Distinct records with different counts: 0

Merge classification:
_merge
both          22202
left_only         0
right_only        0
Name: count, dtype: int64

RESULT: Overlapping records are identical, including multiplicities.

BuyerPulse overlap validation PASSED!


### Cross-Year Overlap — Final Validated Finding

A multiplicity-aware comparison confirmed that the two UCI Online Retail II worksheets contain identical transaction records during December 1–9, 2010.

**Validation results:**

- First worksheet: 22,523 overlapping rows.
- Second worksheet: 22,523 overlapping rows.
- Distinct matching record patterns: 22,202.
- Unmatched record patterns: 0.
- Record multiplicity differences: 0.

The comparison grouped records by all eight original transaction columns, including missing values, then matched their occurrence counts between worksheets.

**Final cleaning decision:** Retain the overlapping period from `Year 2009-2010` and remove the duplicate period from `Year 2010-2011`.

**Expected dataset size after overlap removal:** 1,044,848 rows.

This approach preserves legitimate or potentially legitimate repeated invoice lines within the retained worksheet while eliminating the duplicated cross-year period.

**Validation status:** Passed. Implementation remains part of the cleaning pipeline.

In [32]:
# Step 2.9.1 — Consolidated Data Quality Report
# All statistics are calculated from the original combined data.

import pandas as pd

total_rows = len(retail_df)

invoice_text = (
    retail_df["Invoice"]
    .astype("string")
    .str.strip()
    .str.upper()
)

metrics = {
    "Total transaction rows": total_rows,
    "Missing Customer ID": retail_df["Customer ID"].isna().sum(),
    "Missing Description": retail_df["Description"].isna().sum(),
    "Cancelled invoice rows": invoice_text.str.startswith(
        "C", na=False
    ).sum(),
    "Negative quantity": (retail_df["Quantity"] < 0).sum(),
    "Zero quantity": (retail_df["Quantity"] == 0).sum(),
    "Negative price": (retail_df["Price"] < 0).sum(),
    "Zero price": (retail_df["Price"] == 0).sum(),
    "Exact duplicates within worksheets": retail_df.duplicated(
        subset=list(retail_df.columns),
        keep="first"
    ).sum(),
    "Redundant cross-year overlap rows": 22523,
}

quality_report = pd.DataFrame(
    [
        {
            "Metric": name,
            "Count": int(count),
            "Percentage (%)": round(count / total_rows * 100, 2)
        }
        for name, count in metrics.items()
    ]
)

print("BUYERPULSE — CONSOLIDATED DATA QUALITY REPORT")
print("=" * 55)

display(quality_report)

print("\nIMPORTANT:")
print("These categories overlap and must not be summed.")
print("No cleaning has been applied.")

BUYERPULSE — CONSOLIDATED DATA QUALITY REPORT


,Metric,Count,Percentage (%)
0,Total transaction rows,1067371,100.00
1,Missing Customer ID,243007,22.77
2,Missing Description,4382,0.41
3,Cancelled invoice rows,19494,1.83
4,Negative quantity,22950,2.15
5,Zero quantity,0,0.00
6,Negative price,5,0.00
7,Zero price,6202,0.58
8,Exact duplicates within worksheets,12133,1.14
9,Redundant cross-year overlap rows,22523,2.11



IMPORTANT:
These categories overlap and must not be summed.
No cleaning has been applied.


In [33]:
# Step 2.9.2 — Export Consolidated Data Quality Report

from pathlib import Path

# Resolve the repository root from either the notebook
# directory or the repository root.
cwd = Path.cwd()
project_root = cwd.parent if cwd.name == "notebooks" else cwd

output_dir = project_root / "reports" / "tables"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "data_quality_summary.csv"

quality_report.to_csv(output_path, index=False)

print("BUYERPULSE — REPORT EXPORTED")
print("=" * 45)
print(f"Saved to: {output_path}")
print(f"Rows exported: {len(quality_report)}")

# Verify the exported values match our DataFrame
saved_report = pd.read_csv(output_path)

assert len(saved_report) == len(quality_report)
assert saved_report["Count"].sum() == quality_report["Count"].sum()

print("CSV export validation PASSED!")

BUYERPULSE — REPORT EXPORTED
Saved to: /Users/aymane/Documents/UIR/BuyerPulse/buyerpulse/reports/tables/data_quality_summary.csv
Rows exported: 10
CSV export validation PASSED!
